# G07: Running Your Own Investigation

**Prerequisites:** G01–G06 (the full tutorial sequence).

**What you'll learn:**
- How to formulate an interpretability research question
- The complete workflow from question to finding
- A worked example: does GPT-2 implement subject-verb number agreement?
- How to write up and communicate your results
- Ten open questions you can investigate next

You've learned the tools. Now it's time to use them on a question nobody has answered for you. This tutorial walks through a complete investigation, then sets you free.

---

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.logit_lens import logit_lens
from irtk.patching import patch_by_layer, patch_by_head, ablate_heads, make_logit_diff_metric
from irtk.circuits import direct_logit_attribution, residual_stream_attribution
from irtk.head_analysis import classify_heads
from irtk.neurons import neuron_attribution

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"GPT-2 Small: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads/layer, "
      f"d_model={model.cfg.d_model}")

## The Research Process

Every interpretability investigation follows the same arc. Here's the workflow we'll use:

1. **Pick a question** — something the model does that you want to understand.
2. **Define the task** — create input-output pairs that isolate the behavior.
3. **Define the metric** — a number that captures whether the model does the right thing.
4. **Observe** — verify the model actually shows the behavior.
5. **Locate** — use patching and ablation to find which components matter.
6. **Inspect** — examine those components to understand *how* they do it.
7. **Verify and scope** — test edge cases and limitations.

Steps 1–3 are about designing the experiment. Steps 4–6 are the core investigation. Step 7 keeps you honest. Let's go through all of them on a real question.

## Step 1 — The Question

Our question: **Does GPT-2 implement subject-verb number agreement, and if so, how?**

In English, verbs must agree with their subjects in number:
- "The cat **sits** on the mat" (singular subject → singular verb)
- "The cats **sit** on the mat" (plural subject → plural verb)

This is a clean test case because the correct verb form depends entirely on the subject's number. If GPT-2 gets this right, *something* inside it must be tracking whether the subject is singular or plural and routing that information to the verb prediction. What is that something?

## Step 2 — The Task

We need minimal pairs: sentences that are identical except for the subject's number. For each pair, the correct verb form flips. This isolates number agreement from everything else.

In [ ]:
# Minimal pairs for subject-verb number agreement
test_cases = [
    {
        "singular": "The cat",
        "plural": "The cats",
        "verb_singular": " sits",
        "verb_plural": " sit",
        "context": " on the mat",
    },
    {
        "singular": "The dog",
        "plural": "The dogs",
        "verb_singular": " runs",
        "verb_plural": " run",
        "context": " in the park",
    },
    {
        "singular": "The boy",
        "plural": "The boys",
        "verb_singular": " plays",
        "verb_plural": " play",
        "context": " with toys",
    },
    {
        "singular": "The girl",
        "plural": "The girls",
        "verb_singular": " sings",
        "verb_plural": " sing",
        "context": " in the choir",
    },
    {
        "singular": "The bird",
        "plural": "The birds",
        "verb_singular": " flies",
        "verb_plural": " fly",
        "context": " over the trees",
    },
    {
        "singular": "The teacher",
        "plural": "The teachers",
        "verb_singular": " speaks",
        "verb_plural": " speak",
        "context": " to the class",
    },
    {
        "singular": "The horse",
        "plural": "The horses",
        "verb_singular": " runs",
        "verb_plural": " run",
        "context": " across the field",
    },
]

# Harder cases: intervening clauses that introduce distractor nouns
hard_cases = [
    {
        "singular": "The cat that chased the dogs",
        "plural": "The cats that chased the dog",
        "verb_singular": " sits",
        "verb_plural": " sit",
        "context": " on the mat",
    },
    {
        "singular": "The boy near the tall trees",
        "plural": "The boys near the tall tree",
        "verb_singular": " plays",
        "verb_plural": " play",
        "context": " outside",
    },
    {
        "singular": "The teacher who knows the students",
        "plural": "The teachers who know the student",
        "verb_singular": " speaks",
        "verb_plural": " speak",
        "context": " clearly",
    },
]

# Verify tokenization: make sure each verb is a single token
print("Verifying verb tokenization (each verb should be a single token):")
for case in test_cases + hard_cases:
    v_sg = tokenizer.encode(case["verb_singular"])
    v_pl = tokenizer.encode(case["verb_plural"])
    print(f"  {case['verb_singular']!r}: {v_sg} ({len(v_sg)} tok)  |  "
          f"{case['verb_plural']!r}: {v_pl} ({len(v_pl)} tok)")

## Step 3 — The Metric

We need a single number that captures whether the model gets number agreement right. The natural choice is **logit difference**: logit(correct verb) − logit(wrong verb). For singular subjects, the correct verb is "sits" and the wrong verb is "sit"; for plural subjects, vice versa.

- Positive logit diff → model prefers the correct form
- Negative logit diff → model prefers the wrong form
- Magnitude → how confident the model is

In [ ]:
# Step 4 (Observe): Does GPT-2 actually do number agreement?
print("BASELINE: Does GPT-2 prefer the correct verb form?")
print("=" * 75)
print(f"{'Subject':<40} {'Correct':>8} {'Wrong':>8} {'Diff':>8} {'OK?':>5}")
print("-" * 75)

all_diffs = []

for label, cases in [("Simple", test_cases), ("Hard (distractors)", hard_cases)]:
    diffs_group = []
    print(f"\n  --- {label} ---")
    for case in cases:
        for form in ["singular", "plural"]:
            subject = case[form]
            correct_verb = case[f"verb_{form}"]
            wrong_form = "plural" if form == "singular" else "singular"
            wrong_verb = case[f"verb_{wrong_form}"]

            tokens = model.to_tokens(subject)
            logits = model(tokens)

            correct_id = tokenizer.encode(correct_verb)[0]
            wrong_id = tokenizer.encode(wrong_verb)[0]

            correct_logit = float(logits[-1, correct_id])
            wrong_logit = float(logits[-1, wrong_id])
            diff = correct_logit - wrong_logit
            diffs_group.append(diff)
            all_diffs.append(diff)

            ok = "YES" if diff > 0 else "NO"
            print(f"  {subject + ' _':<38} {correct_logit:>8.2f} {wrong_logit:>8.2f} "
                  f"{diff:>+8.2f} {ok:>5}")

    mean_diff = np.mean(diffs_group)
    acc = np.mean([d > 0 for d in diffs_group])
    print(f"  {label} mean diff: {mean_diff:+.2f}, accuracy: {acc:.0%}")

print(f"\nOverall: mean logit diff = {np.mean(all_diffs):+.2f}, "
      f"accuracy = {np.mean([d > 0 for d in all_diffs]):.0%}")

## Interpreting the Baseline

GPT-2 should show a clear preference for the correct verb form in the simple cases. The hard cases (with intervening distractors like "The cat that chased the dogs") may be weaker — the model has to track agreement across a relative clause with a distractor noun of the opposite number.

If the model gets simple cases right, the behavior exists and we can investigate the mechanism. If the hard cases are weaker, that's interesting too — it suggests the circuit has a limited attention span.

## Step 5 — Locate the Circuit

Now we use activation patching to find *where* number agreement happens. The strategy: take a singular sentence (clean run) and a plural sentence (corrupted run). Patch one layer/head at a time from the corrupted run into the clean run. If patching a component flips the model's verb preference, that component is important for agreement.

We'll start coarse (layer-level) then go fine-grained (head-level).

In [ ]:
# Layer-level patching: which layers carry number information?
# We'll average results across test cases for robustness.

all_layer_results = []

for case in test_cases:
    clean_text = case["singular"]  # e.g. "The cat"
    corrupted_text = case["plural"]  # e.g. "The cats"

    clean_tokens = model.to_tokens(clean_text)
    corrupted_tokens = model.to_tokens(corrupted_text)

    # Skip if token lengths don't match (patching requires same shape)
    if len(clean_tokens) != len(corrupted_tokens):
        continue

    correct_id = tokenizer.encode(case["verb_singular"])[0]
    wrong_id = tokenizer.encode(case["verb_plural"])[0]
    metric_fn = make_logit_diff_metric(correct_id, wrong_id)

    layer_results = patch_by_layer(model, clean_tokens, corrupted_tokens, metric_fn)
    all_layer_results.append(layer_results)

if all_layer_results:
    # Normalize: compute the change from clean baseline
    avg_layer_results = np.mean(all_layer_results, axis=0)

    # Get clean baselines for reference
    clean_baselines = []
    for case in test_cases:
        clean_tokens = model.to_tokens(case["singular"])
        corrupted_tokens = model.to_tokens(case["plural"])
        if len(clean_tokens) != len(corrupted_tokens):
            continue
        correct_id = tokenizer.encode(case["verb_singular"])[0]
        wrong_id = tokenizer.encode(case["verb_plural"])[0]
        metric_fn = make_logit_diff_metric(correct_id, wrong_id)
        clean_baselines.append(float(metric_fn(model(clean_tokens))))

    mean_clean = np.mean(clean_baselines)
    # Patching effect: how much does patching this layer move the metric toward corrupted?
    patch_effect = avg_layer_results - mean_clean

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#d62728' if v < 0 else '#2ca02c' for v in patch_effect]
    ax.bar(range(model.cfg.n_layers), patch_effect, color=colors, alpha=0.8)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_xticks(range(model.cfg.n_layers))
    ax.set_xticklabels([f"L{i}" for i in range(model.cfg.n_layers)])
    ax.set_xlabel("Layer")
    ax.set_ylabel("Patching Effect on Logit Diff")
    ax.set_title("Layer-Level Activation Patching (singular \u2192 plural)\n"
                 "Negative = this layer carries number agreement information")
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    # Report the most important layers
    ranked = np.argsort(patch_effect)
    print("Layers most important for number agreement (largest negative effect):")
    for i in ranked[:5]:
        print(f"  Layer {i}: effect = {patch_effect[i]:+.3f}")
else:
    print("No valid test cases with matching token lengths for patching.")

In [ ]:
# Head-level patching: which specific heads carry number information?

all_head_results = []
clean_baselines = []

for case in test_cases:
    clean_tokens = model.to_tokens(case["singular"])
    corrupted_tokens = model.to_tokens(case["plural"])

    if len(clean_tokens) != len(corrupted_tokens):
        continue

    correct_id = tokenizer.encode(case["verb_singular"])[0]
    wrong_id = tokenizer.encode(case["verb_plural"])[0]
    metric_fn = make_logit_diff_metric(correct_id, wrong_id)

    head_results = patch_by_head(model, clean_tokens, corrupted_tokens, metric_fn)
    all_head_results.append(head_results)
    clean_baselines.append(float(metric_fn(model(clean_tokens))))

if all_head_results:
    avg_head_results = np.mean(all_head_results, axis=0)
    mean_clean = np.mean(clean_baselines)
    head_effect = avg_head_results - mean_clean

    fig, ax = plt.subplots(figsize=(14, 6))
    vmax = max(abs(head_effect.min()), abs(head_effect.max()))
    im = ax.imshow(head_effect, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_xticks(range(model.cfg.n_heads))
    ax.set_yticks(range(model.cfg.n_layers))
    ax.set_title("Head-Level Activation Patching for Number Agreement\n"
                 "(Red = patching flips preference, Blue = reinforces it)")
    plt.colorbar(im, ax=ax, label="Patching Effect")

    # Annotate heads with large effects
    threshold = 0.3 * vmax
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            if abs(head_effect[layer, head]) > threshold:
                ax.text(head, layer, f"{head_effect[layer, head]:+.2f}",
                       ha='center', va='center', fontsize=6, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Top heads by absolute effect
    flat_effect = head_effect.flatten()
    ranked = np.argsort(np.abs(flat_effect))[::-1]
    print("Top 10 heads for number agreement (by absolute patching effect):")
    important_heads = []
    for idx in ranked[:10]:
        l = idx // model.cfg.n_heads
        h = idx % model.cfg.n_heads
        important_heads.append((l, h))
        print(f"  L{l}H{h}: effect = {head_effect[l, h]:+.3f}")
else:
    important_heads = []
    print("No valid test cases with matching token lengths for patching.")

## Step 6 — Inspect the Circuit

We've found the heads that matter most. Now we want to understand *what they do*. Two key questions:
1. **Where do they attend?** Do they look at the subject token (where number is marked)?
2. **What do they write?** Do they directly promote the correct verb?

In [ ]:
# Attention patterns of the top impactful heads on a test case
test_text = "The cat"
tokens = model.to_tokens(test_text)
token_labels = [tokenizer.decode([t]) for t in np.array(tokens)]

logits, cache = model.run_with_cache(tokens)

# Show attention patterns for the top heads (up to 6)
heads_to_show = important_heads[:6] if important_heads else [(0, 0)]
n_show = len(heads_to_show)
n_cols = min(3, n_show)
n_rows = (n_show + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False)
for idx, (layer, head) in enumerate(heads_to_show):
    ax = axes[idx // n_cols, idx % n_cols]
    pattern = np.array(cache[("pattern", layer)][head])
    im = ax.imshow(pattern, cmap='Blues', vmin=0)
    ax.set_title(f"L{layer}H{head}", fontsize=11)
    ax.set_xticks(range(len(token_labels)))
    ax.set_xticklabels(token_labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(token_labels)))
    ax.set_yticklabels(token_labels, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)

# Hide unused subplots
for idx in range(n_show, n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.suptitle(f"Attention Patterns of Top Agreement Heads on \"{test_text}\"", fontsize=13)
plt.tight_layout()
plt.show()

# For each head, report: does the last position attend to the subject token?
print(f"\nDoes the last position attend to the subject token?")
for layer, head in heads_to_show:
    pattern = np.array(cache[("pattern", layer)][head])
    # Last row = attention from last position; look at attention to each token
    last_row = pattern[-1]
    print(f"  L{layer}H{head}: attention from last pos -> {dict(zip(token_labels, [f'{a:.3f}' for a in last_row]))}")

In [ ]:
# Direct logit attribution: which heads directly push verb logits?
test_text = "The cat"
tokens = model.to_tokens(test_text)
logits, cache = model.run_with_cache(tokens)

verb_sg_id = tokenizer.encode(" sits")[0]
verb_pl_id = tokenizer.encode(" sit")[0]

# DLA for the singular verb
dla_sg = direct_logit_attribution(model, cache, token=verb_sg_id, pos=-1)
# DLA for the plural verb
dla_pl = direct_logit_attribution(model, cache, token=verb_pl_id, pos=-1)

# The DLA difference tells us which heads push toward the correct form
dla_diff = dla_sg - dla_pl  # positive = promotes singular over plural

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, data, title in zip(axes, [dla_sg, dla_pl, dla_diff],
                            ['DLA for " sits" (singular)',
                             'DLA for " sit" (plural)',
                             'Difference (singular \u2212 plural)']):
    vmax = max(abs(data.min()), abs(data.max()))
    im = ax.imshow(data, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_xticks(range(model.cfg.n_heads))
    ax.set_yticks(range(model.cfg.n_layers))
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax)

plt.suptitle('Direct Logit Attribution: "The cat" \u2192 verb prediction', fontsize=13)
plt.tight_layout()
plt.show()

# Report heads that most differentially promote singular over plural
flat_diff = dla_diff.flatten()
ranked = np.argsort(flat_diff)[::-1]
print("Heads that most promote 'sits' over 'sit' (singular preference):")
for idx in ranked[:5]:
    l = idx // model.cfg.n_heads
    h = idx % model.cfg.n_heads
    print(f"  L{l}H{h}: DLA diff = {dla_diff[l, h]:+.3f}")

print("\nHeads that most promote 'sit' over 'sits' (plural preference):")
for idx in ranked[-5:][::-1]:
    l = idx // model.cfg.n_heads
    h = idx % model.cfg.n_heads
    print(f"  L{l}H{h}: DLA diff = {dla_diff[l, h]:+.3f}")

In [ ]:
# Residual stream attribution: full decomposition of the verb logit
test_text = "The cat"
tokens = model.to_tokens(test_text)
logits, cache = model.run_with_cache(tokens)

verb_sg_id = tokenizer.encode(" sits")[0]
verb_pl_id = tokenizer.encode(" sit")[0]

attr_sg = residual_stream_attribution(model, cache, token=verb_sg_id, pos=-1)
attr_pl = residual_stream_attribution(model, cache, token=verb_pl_id, pos=-1)

# Compute the difference: which components push toward singular vs plural
all_keys = sorted(attr_sg.keys())
attr_diff = {k: attr_sg[k] - attr_pl[k] for k in all_keys}

# Sort by absolute difference
sorted_keys = sorted(attr_diff.keys(), key=lambda k: abs(attr_diff[k]), reverse=True)

# Plot top 15 components
top_keys = sorted_keys[:15]
top_vals = [attr_diff[k] for k in top_keys]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#d62728' if v > 0 else '#1f77b4' for v in top_vals]
ax.barh(range(len(top_keys)), top_vals, color=colors, alpha=0.8)
ax.set_yticks(range(len(top_keys)))
ax.set_yticklabels(top_keys, fontsize=9)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Logit contribution (singular \u2212 plural)')
ax.set_title('Residual Stream Attribution Difference: which components favor singular?\n'
             'Red = favors singular verb, Blue = favors plural verb')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Top components pushing toward the correct singular form:")
for k in sorted_keys[:5]:
    print(f"  {k}: {attr_diff[k]:+.3f}")

## Step 7 — Verify and Scope

Every finding needs boundary conditions. We've identified heads involved in number agreement on simple cases. Now let's test the limits: do the same components handle agreement when there are distractors? What about longer distances?

In [ ]:
# Edge cases: test the circuit on harder sentences
edge_cases = [
    # Long-distance agreement
    ("The cat that the dogs chased", " sits", " sit", "Long-distance + distractor"),
    # Multiple potential subjects
    ("The owner of the cats", " walks", " walk", "PP attachment"),
    # Simple baseline for comparison
    ("The cat", " sits", " sit", "Simple baseline"),
    # Irregular verbs
    ("The child", " is", " are", "Irregular: is/are"),
    ("The children", " are", " is", "Irregular: are/is (plural)"),
    # Coordination
    ("The cat and the dog", " sit", " sits", "Coordination (plural subject)"),
]

print("Edge Case Analysis")
print("=" * 80)
print(f"{'Case':<32} {'Subject':<30} {'Diff':>8} {'Correct?':>10}")
print("-" * 80)

for subject, correct_verb, wrong_verb, label in edge_cases:
    tokens = model.to_tokens(subject)
    logits = model(tokens)

    correct_id = tokenizer.encode(correct_verb)[0]
    wrong_id = tokenizer.encode(wrong_verb)[0]

    correct_logit = float(logits[-1, correct_id])
    wrong_logit = float(logits[-1, wrong_id])
    diff = correct_logit - wrong_logit

    ok = "YES" if diff > 0 else "NO"
    print(f"  {label:<30} {subject:<30} {diff:>+8.2f} {ok:>10}")

print("\nNote: positive diff = model prefers correct verb form.")
print("Larger magnitude = higher confidence.")

In [ ]:
# Ablation verification: ablate the top heads and check if agreement degrades
test_text = "The cat"
tokens = model.to_tokens(test_text)

verb_sg_id = tokenizer.encode(" sits")[0]
verb_pl_id = tokenizer.encode(" sit")[0]
metric_fn = make_logit_diff_metric(verb_sg_id, verb_pl_id)

# Baseline metric
clean_logits = model(tokens)
baseline = float(metric_fn(clean_logits))

# Ablate all heads and measure effect
ablation_results = ablate_heads(model, tokens, metric_fn, method="zero")

# Effect of ablating each head
ablation_effect = ablation_results - baseline

fig, ax = plt.subplots(figsize=(14, 6))
vmax = max(abs(ablation_effect.min()), abs(ablation_effect.max()))
im = ax.imshow(ablation_effect, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_xticks(range(model.cfg.n_heads))
ax.set_yticks(range(model.cfg.n_layers))
ax.set_title(f'Head Ablation Effect on Number Agreement\n'
             f'\"The cat\" \u2192 \" sits\" vs \" sit\" (baseline diff = {baseline:.2f})\n'
             f'Red = ablation hurts agreement, Blue = ablation helps')
plt.colorbar(im, ax=ax, label="Change in Logit Diff")
plt.tight_layout()
plt.show()

# Report heads whose ablation most hurts agreement
flat = ablation_effect.flatten()
ranked = np.argsort(flat)
print(f"Baseline logit diff (sits - sit): {baseline:.3f}")
print(f"\nHeads whose ablation most HURTS agreement (largest negative effect):")
for idx in ranked[:8]:
    l = idx // model.cfg.n_heads
    h = idx % model.cfg.n_heads
    print(f"  L{l}H{h}: effect = {ablation_effect[l, h]:+.3f} "
          f"(metric after ablation: {ablation_results[l, h]:.3f})")

## Writing It Up

Every interpretability finding should be communicable. Here's a template:

```
## Finding: [One-sentence summary]

### Task
[What behavior did you study?]

### Metric
[How did you measure the behavior?]

### Key Components
[Which heads/layers are involved?]

### Mechanism
[How do the components implement the behavior?]

### Scope and Limitations
[When does it work? When does it fail?]

### Open Questions
[What don't you know yet?]
```

The best write-ups are honest about what was found and what wasn't. Negative results (“the model *doesn’t* do X” or “the circuit breaks in case Y”) are just as valuable as positive ones.

In [ ]:
# Our write-up, assembled from the investigation above.
# We'll gather the key facts programmatically.

# Re-run baseline to get summary stats
simple_diffs = []
for case in test_cases:
    for form in ["singular", "plural"]:
        subject = case[form]
        correct_verb = case[f"verb_{form}"]
        wrong_form = "plural" if form == "singular" else "singular"
        wrong_verb = case[f"verb_{wrong_form}"]
        tokens = model.to_tokens(subject)
        logits = model(tokens)
        correct_id = tokenizer.encode(correct_verb)[0]
        wrong_id = tokenizer.encode(wrong_verb)[0]
        diff = float(logits[-1, correct_id]) - float(logits[-1, wrong_id])
        simple_diffs.append(diff)

hard_diffs = []
for case in hard_cases:
    for form in ["singular", "plural"]:
        subject = case[form]
        correct_verb = case[f"verb_{form}"]
        wrong_form = "plural" if form == "singular" else "singular"
        wrong_verb = case[f"verb_{wrong_form}"]
        tokens = model.to_tokens(subject)
        logits = model(tokens)
        correct_id = tokenizer.encode(correct_verb)[0]
        wrong_id = tokenizer.encode(wrong_verb)[0]
        diff = float(logits[-1, correct_id]) - float(logits[-1, wrong_id])
        hard_diffs.append(diff)

# Summary
print("=" * 70)
print("FINDING: GPT-2 implements subject-verb number agreement")
print("=" * 70)

print(f"""
TASK
  Subject-verb number agreement: does GPT-2 prefer the correct
  verb form (singular vs plural) to match the subject's number?

METRIC
  Logit difference: logit(correct_verb) - logit(wrong_verb).
  Positive = correct preference.

BASELINE
  Simple cases: {np.mean([d > 0 for d in simple_diffs]):.0%} accuracy, mean diff = {np.mean(simple_diffs):+.2f}
  Hard cases (distractors): {np.mean([d > 0 for d in hard_diffs]):.0%} accuracy, mean diff = {np.mean(hard_diffs):+.2f}

KEY COMPONENTS""")

if important_heads:
    print("  Top heads by patching effect:")
    for l, h in important_heads[:5]:
        print(f"    L{l}H{h}")

print(f"""
SCOPE AND LIMITATIONS
  - Works well on simple "The X verb" patterns
  - May degrade with intervening clauses containing distractor nouns
  - Performance varies by verb pair and subject

OPEN QUESTIONS
  - Is number encoded as a linear direction in residual stream space?
  - Do the same heads handle agreement for all verb forms?
  - How does this interact with the induction circuit?
""")

## Ten Open Questions

You now have the full toolkit and the methodology. Here are ten concrete research questions you can investigate with IRTK. Each one is tractable — you could make progress on any of them in an afternoon.

### 1. Induction Head Composition
How exactly do previous-token heads compose with induction heads? Can you trace the information flow through the residual stream?

**Relevant modules:** `irtk.head_analysis` (find_induction_heads, find_previous_token_heads), `irtk.circuits` (ov_circuit, qk_circuit), `irtk.patching` (patch_by_head)

### 2. Factual Knowledge Localization
Which specific MLP neurons store the fact "Paris is the capital of France"? Can you edit the fact by modifying those neurons?

**Relevant modules:** `irtk.neurons` (neuron_attribution, top_activating_tokens), `irtk.knowledge` (knowledge_neurons, causal_tracing), `irtk.model_editing` (rank_one_edit)

### 3. Sentiment Circuits
How does GPT-2 determine the sentiment of a sentence? Is there a "sentiment direction" in activation space that you can find with probing?

**Relevant modules:** `irtk.probes` (LinearProbe, train_linear_probe), `irtk.concept_erasure` (concept_directions), `irtk.representation_engineering` (reading_vector)

### 4. Copy Suppression
Some attention heads actively suppress copying. When and why do they fire? Can you find them with negative direct logit attribution?

**Relevant modules:** `irtk.copy_suppression` (copy_suppress_decomposition), `irtk.circuits` (direct_logit_attribution, full_ov_circuit), `irtk.attention_utils`

### 5. Gendered Pronoun Resolution
How does GPT-2 resolve "he" vs "she" in context? Is there a specific circuit, and does it rely on attention to the antecedent?

**Relevant modules:** `irtk.patching` (patch_by_head), `irtk.circuits` (direct_logit_attribution), `irtk.binding_analysis` (binding_attention)

### 6. Attention Sink Analysis
GPT-2 often assigns high attention to the BOS token or punctuation. Why? Is this meaningful computation or a dumping ground for unused attention mass?

**Relevant modules:** `irtk.attention_sink_analysis` (sink_identification, sink_heads), `irtk.attention_utils` (entropy), `irtk.patching` (ablate_heads)

### 7. MLP vs Attention Contribution
For a specific task (say, factual recall), what fraction of the work is done by attention vs MLPs? Does this ratio change across layers?

**Relevant modules:** `irtk.circuits` (residual_stream_attribution), `irtk.prediction_decomposition`, `irtk.layer_composition`

### 8. Feature Universality
Train sparse autoencoders on different layers. Do the same features appear across multiple layers? How do features transform as you go deeper?

**Relevant modules:** `irtk.sae` (SparseAutoencoder, train_sae), `irtk.sparse_features` (feature_correlation), `irtk.cross_layer_feature_tracking`

### 9. Training Dynamics
Use `irtk.training` to train a tiny model on an algorithmic task. When do induction heads appear? What causes the phase transition?

**Relevant modules:** `irtk.training` (train_tiny_model), `irtk.developmental_interpretability` (phase_transitions, circuit_formation), `irtk.head_analysis`

### 10. Backup Circuits
If you ablate the "main" circuit for a task (e.g., the top heads for IOI), does a backup circuit take over? How much performance is recovered?

**Relevant modules:** `irtk.backup_detection` (backup_head_detection, knockout_compensation), `irtk.patching` (ablate_heads), `irtk.experiments` (find_circuit)

## Final Thoughts

Mechanistic interpretability is a young field. The techniques you've learned in this tutorial series — logit lens, activation patching, direct logit attribution, circuit analysis, probing, SAEs — are the standard toolkit, but the questions are wide open.

The most impactful discoveries in mechinterp have come from people who picked a concrete behavior, measured it carefully, and patiently traced the mechanism. Induction heads, indirect object identification, copy suppression — all of these started as "I wonder how the model does *this specific thing*."

You have the tools. You have the methodology. Go find something interesting.

## Further Reading

- Neel Nanda, ["200 Concrete Open Problems in Mechanistic Interpretability"](https://www.alignmentforum.org/s/yivyHaCAmMJ3CqSyj) — a rich source of tractable research questions.
- Olah et al. 2020, ["Zoom In: An Introduction to Circuits"](https://distill.pub/2020/circuits/zoom-in/) — the foundational vision for circuits-level interpretability.
- Linzen et al. 2016, ["Assessing the Ability of LSTMs to Learn Syntax-Sensitive Dependencies"](https://arxiv.org/abs/1611.01368) — the original number agreement test for neural LMs.
- IRTK API cheatsheet: `notebooks/00_api_cheatsheet.ipynb`